# Cross-subject ERS/ERD components by non-negative CP decomposition

A single worked run of the whole pipeline, end to end: load per-subject time-frequency
power, split it into synchronisation (ERS) and desynchronisation (ERD), decompose each
subject independently with non-negative CP, then work out which components describe the
same phenomenon in different subjects.

It runs on **simulated data with known ground truth**, so it works from a fresh clone with
no recordings, and so the last section can check the recovered clusters against the
patterns that actually went in. Section 2 marks the one line to change to run it on real
data.

The method, and the reasoning behind each choice, is in [`docs/pipeline.md`](../docs/pipeline.md).

## 1. Setup

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
import numpy as np
import matplotlib.pyplot as plt

# Remove this line once the package is installed (`pip install -e .` from the repo root).
sys.path.insert(0, '../src')

import opm_cp as cp
from opm_cp import plotting as vz
from opm_cp import simulate

plt.rcParams['figure.dpi'] = 110

## 2. Config

In [ ]:
SUBJECTS = [1, 2, 3]
R = 3                      # CP rank per subject, per tensor type — justified in section 4
N_CROP = 5                 # time points dropped each end (time-frequency edge artefacts)
LAG_MS = 150               # latency difference tolerated between matching time courses
TYPES = ('ERS', 'ERD')

# Cortical parcels the OPM array actually covers. Matching is case-insensitive substring
# matching, so 'occipital' keeps every occipital parcel in both hemispheres.
KEEP_PATTERNS = [
    # Frontal / motor
    'precentral', 'frontal_sup_2', 'frontal_mid_2', 'supp_motor_area',
    'frontal_sup_medial', 'rolandic_oper',
    # Occipital / ventral visual
    'calcarine', 'cuneus', 'lingual', 'occipital_sup', 'occipital_mid', 'occipital_inf',
    'fusiform',
    # Parietal
    'postcentral', 'parietal_sup', 'parietal_inf', 'supramarginal', 'angular', 'precuneus',
    # Temporal
    'heschl', 'temporal_sup', 'temporal_mid', 'temporal_inf',
]

# The AAL2 atlas is not bundled (see atlas/README.md). Without it the ROI mode is drawn
# as a labelled bar panel instead of a glass brain; nothing else changes.
try:
    ATLAS = cp.load_aal_atlas('../atlas/aal2.nii', '../atlas/aal2.nii.txt')
except Exception as e:
    ATLAS = None
    print(f'No atlas ({type(e).__name__}) — ROI panels will be bar plots. See atlas/README.md')

### Load the data

**This is the only cell to change for real data.** Replace the `simulate_dataset` call
with your own files — the dicts have the same keys either way:

```python
MAT = '../data/TFR_Subject{s}_induced.mat'
loaded = {s: cp.load_tensor(MAT.format(s=s)) for s in SUBJECTS}
```

In [ ]:
loaded = simulate.simulate_dataset(subjects=SUBJECTS, verbose=True)

`build_dataset` then does the three preparation steps — keep the cortical parcels, crop
the time-frequency edges, and split the signed log-ratio power into its positive part
(ERS) and the magnitude of its negative part (ERD). The two are decomposed separately,
because a power increase and a power decrease in the same parcel are different phenomena
and should not be forced to share a component.

In [ ]:
ds = cp.build_dataset(loaded, keep_patterns=KEEP_PATTERNS, n_crop=N_CROP)
COLORS = vz.subject_colors(ds.subjects)     # one colour per subject, used by every figure

### A look at the undecomposed data

Worth doing before and after: a CP component should correspond to something visible here.
Red is power above baseline (ERS), blue below (ERD).

In [ ]:
_ = vz.plot_tfr_roi(loaded[SUBJECTS[0]]['pow_norm'], loaded[SUBJECTS[0]]['roi_labels'],
                    ['precentral_l', 'occipital_mid_l'],
                    loaded[SUBJECTS[0]]['freq_vec'], loaded[SUBJECTS[0]]['time_vec'],
                    flim=(5, 50), title=f'Subject {SUBJECTS[0]} — baseline-normalised power')

## 3. Choosing a rank

R² rises monotonically with rank, so it cannot pick a rank on its own. Pair it with
**stability**: refit the same rank from many initialisations and measure how reproducible
the components are. Stability holds near 1 while the rank is supported by the data and
collapses once it is not, so take the largest rank that is still both stable and explains
a useful share of the variance.

Run this on one subject and one tensor type to keep it quick; for a real analysis, sweep
every subject and check the choice is not driven by one of them.

In [ ]:
rank_results = cp.rank_selection(cp.as_tensor(ds.tensors['ERD'][SUBJECTS[0]]),
                                 ranks=range(1, 7), n_repeats=5, verbose=0)
vz.plot_rank_selection(rank_results, title_prefix=f'ERD, subject {SUBJECTS[0]}:')

for rank, stab, r2 in zip(rank_results['ranks'], rank_results['mean_stabilities'],
                          rank_results['best_r2']):
    print(f'  rank {rank}: stability {stab:.3f}   R² {r2:.3f}')

## 4. Decompose each subject

Every subject is decomposed independently — no spatial or temporal correspondence between
subjects is assumed at this stage. Each fit is the best of `n_runs` random
initialisations, because the CP objective is non-convex and a single fit lands wherever
its starting point leads; with too few restarts a reported component can be an artefact
of one unlucky initialisation. Raise `n_runs` to 25 for anything you intend to report.

In [ ]:
cp_models, factors = cp.decompose_dataset(ds, rank=R, n_runs=10, n_iter_max=2500,
                                          verbose=False)

for t in TYPES:
    r2s = [cp.r_squared(cp.as_tensor(ds.tensors[t][s]), cp_models[t][s]) for s in ds.subjects]
    print(f'{t}: factor shapes {[f.shape for f in factors[t][ds.subjects[0]]]}  |  '
          f'R² per subject ' + ', '.join(f'S{s}={r:.3f}' for s, r in zip(ds.subjects, r2s)))

### Do the subjects' models resemble each other at all?

The coarse version of the question the clustering answers properly: whole-model
similarity, with the components Hungarian-matched between each pair of subjects. Low
values here mean the clustering will have little to find.

In [ ]:
vz.plot_model_similarity(cp_models, ds.subjects, cp.cosine_similarity_score, types=TYPES)

### One subject's factors

The raw factors as fitted, before any cross-subject work. Change `subject` and `t` to
look at another decomposition; `plot_component` gives the fuller single-component view
with the top parcels named.

In [ ]:
subject, t = SUBJECTS[0], 'ERD'
vz.plot_cp3_full(factors[t][subject], ds.freq_vec, ds.time_vec, plot_type=t)

In [ ]:
# One component in full: frequency and time profiles, the ROI mode, and the top parcels.
vz.plot_component(factors[t][subject], 0, ds.roi_labels, ds.freq_vec, ds.time_vec,
                  atlas=ATLAS, plot_type=t, top_n=8,
                  title=f'{t} — subject {subject}, component 1')

## 5. Cross-subject component similarity

All subjects' components are flattened into one list and every pair is scored. The score
is the **product** of the per-mode similarities, so a pair must agree in anatomy, in
frequency *and* in time — a shared rhythm in unrelated parcels is not a shared component.

ROI and frequency are compared by cosine. The time mode is compared by the **maximum
normalised cross-correlation over lags within ±`LAG_MS`**, because the same event does
not occur at the same latency in two subjects: scored at zero lag, two genuinely matching
components offset by 100 ms score poorly and clusters fragment into singletons. The
cross-correlation stays in [0, 1], so it remains comparable to the two cosines, but it no
longer penalises a shift the experiment never controlled.

In [ ]:
sims = {}
for t in TYPES:
    print(f'--- {t} ---')
    components = cp.build_component_set(factors[t], subjects=ds.subjects, rank=R,
                                        mode_names=ds.mode_names)
    sims[t] = cp.compute_similarity(components, dt_ms=ds.dt_ms, lag_ms=LAG_MS,
                                    time_weight=1.0)

Before imposing any clustering, look at the whole similarity structure. Each component is
a node on a ring, split into one sector per subject; each pair is a chord weighted by
similarity, with cross-subject chords in black and on top. Comparing the per-mode panels
shows which mode is carrying the cross-subject structure.

In [ ]:
vz.plot_connectogram_grid(sims, COLORS, gamma=3.0)

## 6. Grouping the components

Two methods. Both number clusters 1..K, with **0 meaning unassigned**.

**Greedy alignment** (`align_components`) seeds slots from one reference subject and
matches the others onto them one-to-one, opening a new slot when nothing clears the
threshold. Fast and readable, but it depends on which subject seeds the slots, and the
one-to-one constraint can force a poor match when a subject genuinely lacks a pattern.
Use it to get oriented.

In [ ]:
ALIGN_THRESHOLD = 0.5
N_SLOTS = 5                 # may exceed R, letting subjects contribute patterns the
                            # reference subject does not have

align_clusters = {}
for t in TYPES:
    align_clusters[t] = cp.align_components(sims[t], n_slots=N_SLOTS,
                                            threshold=ALIGN_THRESHOLD)
    print(cp.format_clusters(sims[t], align_clusters[t],
                             f'{t}: aligned slots (n_slots={N_SLOTS}, '
                             f'threshold={ALIGN_THRESHOLD})'))

In [ ]:
# A good clustering shows bright blocks on the diagonal and dark elsewhere. Tick labels
# are coloured by subject, so it is obvious whether a bright block spans subjects or is
# one subject's components matching each other.
for t in TYPES:
    vz.plot_similarity_matrices(sims[t], align_clusters[t], COLORS, title=t)

**Consensus clustering** (`consensus_clustering`) is the result to report. Each clustering
is solved as an integer program that maximises total within-cluster similarity subject to
three constraints: pairs below the threshold cannot be linked, two components of the same
subject cannot share a cluster, and the relation must be transitive so the result is a
genuine partition. That is globally optimal, so no reference subject is needed — and
because it is exact, the stability check has to perturb the *input*: add noise to the
similarity matrix, re-solve, and average how often each pair ends up together
(Monti et al. 2003).

The threshold is on the same scale as the similarities, so set it against their spread —
the median cross-subject similarity printed in section 5 is the right reference. Raise
`n_resamples` to 500 for a result you intend to report.

Needs a MILP solver: `pip install "opm-cp[consensus]"`.

In [ ]:
CONSENSUS_THRESHOLD = 0.5
H = 200                     # perturbed clusterings to average
NOISE_SD = 0.05

consensus_matrix, consensus_clusters, cluster_consensus = {}, {}, {}
for t in TYPES:
    print(f'--- {t} ---')
    consensus_matrix[t], consensus_clusters[t], cluster_consensus[t] = \
        cp.consensus_clustering(sims[t], threshold=CONSENSUS_THRESHOLD, n_resamples=H,
                                noise_sd=NOISE_SD, seed=0)

In [ ]:
# Solid red blocks are clusters that survived every perturbation; pale fractional entries
# are pairings that only sometimes held, i.e. the clustering was near a tie there.
for t in TYPES:
    vz.plot_consensus_matrix(consensus_matrix[t], sims[t], consensus_clusters[t], COLORS,
                             title=f'{t}: consensus matrix ({H} perturbed runs)')

## 7. The clusters

Set `CLUSTERS` to `align_clusters` to plot the alignment slots instead.

Two figures per cluster, deliberately. `plot_cluster` shows **every member separately** —
all members' frequency and time profiles overlaid, coloured by subject, then one ROI panel
each. This is the figure for judging whether a cluster is real: if the profiles genuinely
overlap, it is.

In [ ]:
CLUSTERS = consensus_clusters

for t in TYPES:
    for k in cp.cluster_ids(CLUSTERS[t]):
        vz.plot_cluster(sims[t], CLUSTERS[t], k, ds.roi_labels, ds.freq_vec, ds.time_vec,
                        COLORS, atlas=ATLAS, plot_type=t, threshold_percentile=80)

`plot_cluster_mean` collapses the same cluster to one consensus profile. Three choices
there are worth stating, because at two or three members a shaded band is easy to
over-read:

- **The band is the member min-max range, not ±1 SD.** A sample SD from two points is just
  `|a−b|/√2`, a range wearing a distribution's clothes. The members are drawn over the mean
  so the figure shows the spread it actually has. A range of non-negative loadings also
  cannot dip below zero, which a symmetric mean ± SD band does at low-loading bins.
- **Time courses are lag-aligned before averaging**, onto the cluster medoid and within the
  same ±`LAG_MS` the clustering tolerates. Unaligned, an inter-subject latency shift smears
  the mean and comes back as band width. Samples shifted in from outside the recording are
  left NaN rather than invented, so the band stops at those edges. Pass `align_time=False`
  to compare.
- **Members are max-normalised**, so the panels read on the same 0–1 scale as above. Note
  that this pins every member to 1 at its own peak, so the band is narrow near the peaks
  *by construction* and most informative off-peak.

In [ ]:
for t in TYPES:
    for k in cp.cluster_ids(CLUSTERS[t]):
        vz.plot_cluster_mean(sims[t], CLUSTERS[t], k, ds.roi_labels, ds.freq_vec,
                             ds.time_vec, COLORS, atlas=ATLAS, plot_type=t, top_n=5,
                             align_time=True)

## 8. Cluster quality

Per-mode diagnostics, flagging two specific failures:

- **A weak ROI score alongside high frequency and time scores** usually means a forced
  one-to-one match — an occipital component sitting in a motor cluster because something
  had to go there.
- **A subject whose best-matching component is not the one in the cluster.** Each of that
  subject's components is scored against the template built from the other members. If the
  best scorer sits elsewhere, the assignment is suspect; if none of them score well, that
  subject simply does not express the pattern — a finding, not a bug.

In [ ]:
for t in TYPES:
    print(f'\n{"=" * 20} {t} {"=" * 20}')
    print(cp.cluster_quality(sims[t], CLUSTERS[t]))

## 9. Did it recover the ground truth?

The payoff of running on simulated data. Every shared pattern should come back as one
cluster holding one component per subject, and every subject-specific pattern should be
left unassigned — no cluster should mix two different ground-truth patterns.

Match the clusters to the table below by their peak frequency and latency, read off the
cluster-mean figures in section 7. The per-subject latency shifts in the table are what
the lag-tolerant time similarity had to absorb; `plot_cluster_mean` prints the shift it
recovered for each member, which should be close to the difference between their true
lags.

A cluster with fewer members than there are subjects is worth chasing down rather than
dismissing: compare that subject's `amp` for the pattern in the table below. Where it is
small, the pattern was genuinely weak in that subject and a rank-`R` fit had better uses
for a component — the cluster quality report in section 8 flags exactly this, and the
honest reading is that the subject does not express the pattern, not that the clustering
failed.

On real data there is no table to check against — which is why the consensus `m(k)`
values, the cluster quality report and the member overlap in section 7 are what you lean
on instead.

In [ ]:
print(simulate.truth_table(loaded))

n_shared = sum(t['shared'] for t in loaded[SUBJECTS[0]]['truth'])
print(f'\n{n_shared} shared patterns went in ({len(SUBJECTS)} subjects each). Recovered:')
for t in TYPES:
    ids = cp.cluster_ids(CLUSTERS[t])
    sizes = [int((CLUSTERS[t] == k).sum()) for k in ids]
    n_un = int((CLUSTERS[t] == 0).sum())
    print(f'  {t}: {len(ids)} clusters, sizes {sizes}, {n_un} unassigned  '
          f'| m(k) ' + ', '.join(f'{k}={v:.2f}' for k, v in cluster_consensus[t].items()))